In [5]:
import os  
import asyncio
from io import BytesIO

import pandas as pd
import gspread
from google.oauth2.service_account import Credentials

from datetime import datetime as dt
from datetime import timedelta

from aiogram import Bot, Dispatcher, F, Router
from aiogram.filters import CommandStart, Command
from aiogram.types import Message, CallbackQuery, InlineKeyboardMarkup, InlineKeyboardButton
from aiogram.utils.keyboard import InlineKeyboardBuilder

import matplotlib.pyplot as plt
import pandas as pd

In [6]:
scopes = [
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive'
]

creds = Credentials.from_service_account_file('credentials.json', scopes=scopes)
client = gspread.authorize(creds)

yesterday = dt.today() - timedelta(days=1)
months_dict = {
    1: "Январь",
    2: "Февраль",
    3: "Март",
    4: "Апрель",
    5: "Май",
    6: "Июнь",
    7: "Июль",
    8: "Август",
    9: "Сентябрь",
    10: "Октябрь",
    11: "Ноябрь",
    12: "Декабрь"
}
year = yesterday.year
month = months_dict.get(yesterday.month)
day = yesterday.day

full_date  = yesterday.strftime('%d.%m.%Y')
full_date

'19.09.2026'

In [7]:
key = next((k for k, v in months_dict.items() if v == month), None)
prev_month = months_dict.get(key - 1)
prev_month

'Август'

In [8]:
import re

In [9]:
def get_sample_data():
    sheet = client.open_by_key('1o1mIcsXQht1NFhgsq7CMKI3derC8xOSrRgGC9GYu144').worksheet(f'{month} {year}')
    data = sheet.get_all_values()
    df = pd.DataFrame(data)
    df = df.loc[3:]
    df_basic = df.copy()
    df_basic.columns = df.iloc[0]
    df_basic = df_basic[1:].reset_index(drop=True)
    df_basic = df_basic.set_index('Дата')

    

    return df_basic

In [10]:
def get_ya_metrik(all_data):
    
    return all_data().iloc[:, :10][['номер недели', 'Трафик', 'Уникальные', 'vs LY', 'Переход в каталог', 'Положил в корзину', 'Оформил заказ']]

In [11]:
def get_basic_data(all_data):
    return all_data()[['План Руб', 'План Заказы', 'Заказы Сайт, шт', 'Сумма заказов РУБ']]

In [12]:
def get_reg_info(all_data):
    return all_data().iloc[:, 18:21][[ 'ШТ', 'vs LY']]

In [13]:
def get_cerf_info(all_data):    
    return all_data()[['Сертификаты, шт', 'Сертификаты, Руб']]

In [14]:
type(get_sample_data()['Трафик'].loc['13.09.2026'])

str

In [15]:
df = get_ya_metrik(get_sample_data)
day_ya_info = df.loc[full_date]

day_ya_info = (day_ya_info.astype('str')  
                        .str.replace(r'\s+', '', regex=True)
                        .astype('float32')) # Убираем лишние пробелы во всей таблице и переводим строчные даннын в числовые

In [16]:
day_ya_info


3
номер недели           38.0
Трафик                737.0
Уникальные            578.0
vs LY                2151.0
Переход в каталог     212.0
Положил в корзину      68.0
Оформил заказ          12.0
Name: 19.09.2026, dtype: float32

In [17]:
traffic_vs_LY = 100 * day_ya_info['Уникальные']/day_ya_info['vs LY']
CTR_to_catalog = 100 * day_ya_info['Переход в каталог']/day_ya_info['Уникальные']
CTR_to_basket = 100 * day_ya_info['Положил в корзину']/day_ya_info['Уникальные']
CTR_to_order = 100 * day_ya_info['Оформил заказ']/day_ya_info['Уникальные']
print(f' Доля траффика от прошлогоднего: {traffic_vs_LY:,.2f}% \n \
rerere')


 Доля траффика от прошлогоднего: 26.87% 
 rerere


In [18]:
print(f' Доля траффика от прошлогоднего: {traffic_vs_LY:,.2f}% \n \
    CTR перехода в каталог: {CTR_to_catalog:,.2f}% \n \
    CTR добавления в корзину: {CTR_to_basket:,.2f}% \n \
    CTR создания заказа: {CTR_to_order:,.2f}%')

 Доля траффика от прошлогоднего: 26.87% 
     CTR перехода в каталог: 36.68% 
     CTR добавления в корзину: 11.76% 
     CTR создания заказа: 2.08%


In [19]:
df = get_basic_data(get_sample_data)

day_sales_info = df.loc[full_date]
type(day_sales_info['Сумма заказов РУБ'])

str

In [26]:
df_sales_last5 = df.loc['14.09.2026':f'{full_date}']
df_sales_last5

3,План Руб,План Заказы,"Заказы Сайт, шт",Сумма заказов РУБ
Дата,,,,
14.09.2026,156 667,,33,231 402
15.09.2026,156 667,,25,139 929
16.09.2026,210 000,,35,247 279
17.09.2026,150 000,,,
18.09.2026,150 000,,,
19.09.2026,150 000,,,


In [39]:
df_sales_last5 = df_sales_last5.astype(str).map(lambda x: re.sub(r'\s+', '', x))
df_sales_last5 = df_sales_last5.apply(pd.to_numeric, errors='coerce').astype('float32')

df_sales_last5

3,План Руб,План Заказы,"Заказы Сайт, шт",Сумма заказов РУБ,Доля,Ср. чек
Дата,,,,,,
14.09.2026,156667.0,NaN,33.0,231402.0,1.477031,7012.181641
15.09.2026,156667.0,NaN,25.0,139929.0,0.893162,5597.160156
16.09.2026,210000.0,NaN,35.0,247279.0,1.177519,7065.114258
17.09.2026,150000.0,NaN,NaN,NaN,NaN,NaN
18.09.2026,150000.0,NaN,NaN,NaN,NaN,NaN
19.09.2026,150000.0,NaN,NaN,NaN,NaN,NaN


In [40]:
df_sales_last5['Доля'] = df_sales_last5['Сумма заказов РУБ']/ df_sales_last5['План Руб'] 
df_sales_last5['Ср. чек'] = df_sales_last5['Сумма заказов РУБ']/ df_sales_last5['Заказы Сайт, шт']
df_sales_last5 = df_sales_last5.fillna('нет данных')
df_sales_last5

3,План Руб,План Заказы,"Заказы Сайт, шт",Сумма заказов РУБ,Доля,Ср. чек
Дата,,,,,,
14.09.2026,156667.0,нет данных,33.0,231402.0,1.477031,7012.181641
15.09.2026,156667.0,нет данных,25.0,139929.0,0.893162,5597.160156
16.09.2026,210000.0,нет данных,35.0,247279.0,1.177519,7065.114258
17.09.2026,150000.0,нет данных,нет данных,нет данных,нет данных,нет данных
18.09.2026,150000.0,нет данных,нет данных,нет данных,нет данных,нет данных
19.09.2026,150000.0,нет данных,нет данных,нет данных,нет данных,нет данных


In [41]:
df_sales_last5.index

Index(['14.09.2026', '15.09.2026', '16.09.2026', '17.09.2026', '18.09.2026',
       '19.09.2026'],
      dtype='object', name='Дата')

In [42]:
get_ya_metrik(get_sample_data).loc[df_sales_last5.index]

3,номер недели,Трафик,Уникальные,vs LY,Переход в каталог,Положил в корзину,Оформил заказ
Дата,,,,,,,
14.09.2026,38,991,775,1 900,316,94,30
15.09.2026,38,883,688,2 638,269,73,18
16.09.2026,38,993,788,2 569,303,91,28
17.09.2026,38,848,646,2 635,270,82,22
18.09.2026,38,765,590,2 278,238,67,17
19.09.2026,38,737,578,2 151,212,68,12
